# Layer importance via causal memory-write swapping (appendix)

For each layer, swap the GNM memory write with the untrained model's write and measure the
accuracy/selectivity drop — identifying which layers carry the instruction-conditioned signal.
Produces `plots/appendix_layer_importance.png`.

Inputs: the write-invariance dataset (`data/analysis/counterfact_memory_analysis.json`, shipped) and
the CounterFACT GNM checkpoint (`models/gnm_all/model.pth`).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))


In [ ]:
import random
import json
from collections import defaultdict
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity
from gnm import MemoryLLMChatWrapper, build_model
import torch.nn.functional as F
from instructions import get_learning_instruction
from config_memoryllm_train import MemoryllmTrainConfig, DistributedCfg, DataCfg, SamplingCfg, ModelCfg, LoopCtrlCfg, LoggingCfg
import matplotlib.pyplot as plt

## Dataset construction (run once; the shipped dataset was built with this)

In [ ]:
def find_answer_token_positions(wrapper, text: str, answer: str):
    """
    Find the token position(s) of the answer within a tokenized text.
    Returns list of (start_idx, end_idx) tuples for each occurrence.
    """
    # Tokenize the full text
    text_tokens = wrapper.tokenizer(text, return_tensors="pt", add_special_tokens=False)
    text_ids = text_tokens.input_ids[0].tolist()
    
    # Tokenize just the answer
    answer_tokens = wrapper.tokenizer(answer, return_tensors="pt", add_special_tokens=False)
    answer_ids = answer_tokens.input_ids[0].tolist()
    
    # Also try with leading space (common tokenization pattern)
    answer_with_space = " " + answer
    answer_space_tokens = wrapper.tokenizer(answer_with_space, return_tensors="pt", add_special_tokens=False)
    answer_space_ids = answer_space_tokens.input_ids[0].tolist()
    
    positions = []
    
    # Search for answer tokens in text
    for search_ids in [answer_ids, answer_space_ids]:
        for i in range(len(text_ids) - len(search_ids) + 1):
            if text_ids[i:i+len(search_ids)] == search_ids:
                positions.append((i, i + len(search_ids)))
    
    # Remove duplicates while preserving order
    seen = set()
    unique_positions = []
    for pos in positions:
        if pos not in seen:
            seen.add(pos)
            unique_positions.append(pos)
    
    return unique_positions, text_ids, answer_ids



@torch.no_grad()
def get_memory_delta_per_layer(
    wrapper: MemoryLLMChatWrapper,
    learning_instruction: str,
    document: str,
):
    """
    Returns memory delta for each layer.
    Output shape: [L, num_memory_tokens, hidden_dim]
    """
    wrapper.reset_memory()
    wrapper.model.eval()

    memory_context = (
        f"<|learning_instruction_start|>{learning_instruction}<|learning_instruction_end|>"
        f"<|document_start|>{document}<|document_end|>"
    )

    ctx_ids = wrapper.tokenizer(
        memory_context,
        return_tensors="pt",
        add_special_tokens=False
    ).input_ids.to(wrapper.model.device)

    # Get memory delta without committing
    delta = wrapper.model.inject_memory(ctx_ids, update_memory=False)
    
    # delta is [1, L, num_memory_tokens, hidden_dim]
    return delta[0].detach()  # [L, num_memory_tokens, hidden_dim]



@torch.no_grad()
def get_hidden_states_during_injection(wrapper, learning_instruction: str, document: str):
    """
    Get hidden states at ALL layers during the memory INJECTION forward pass.
    
    IMPORTANT: We call wrapper.model(...) with is_injection=True to get the 
    memory-influenced hidden states, NOT wrapper.model.model(...) which would
    bypass the memory machinery entirely.
    
    Returns: dict with 'hidden_states' tensor of shape [L+1, seq_len, D]
             where seq_len matches the input_ids length (memory tokens are internal)
    """
    wrapper.reset_memory()
    wrapper.model.eval()
    
    memory_context = (
        f"<|learning_instruction_start|>{learning_instruction}<|learning_instruction_end|>"
        f"<|document_start|>{document}<|document_end|>"
    )
    
    ctx_ids = wrapper.tokenizer(
        memory_context,
        return_tensors="pt",
        add_special_tokens=False
    ).input_ids.to(wrapper.model.device)
    
    # CRITICAL: Call the MemoryLLM forward with is_injection=True
    # This runs the actual memory injection forward pass where memory tokens
    # are prepended at each layer. The returned hidden_states correspond to
    # the input_ids positions (memory influence is internal).
    outputs = wrapper.model(
        input_ids=ctx_ids,
        output_hidden_states=True,
        is_injection=True,  # Memory write mode
        return_dict=True
    )
    
    # hidden_states is tuple of (L+1) tensors, each [1, seq_len, D]
    # These are the hidden states for the INPUT tokens (not memory tokens)
    # but they have been computed WITH memory tokens present in attention
    hidden_states = torch.stack([h.squeeze(0) for h in outputs.hidden_states], dim=0)
    
    return {
        'hidden_states': hidden_states.detach(),  # [L+1, seq_len, D]
        'input_ids': ctx_ids.squeeze(0).detach(),  # [seq_len]
        'memory_context': memory_context,  # for debugging
    }

In [ ]:
# # Construct Dataset
# '''
#     dataset = [
#     {
#         "target_fact": "...",
#         "learning_instruction": "...",
#         "examples": [
#             {
#               "document": "...",
#               "distractor_fact": {}
#             },
#             {"document": "...",
#              "distractor_fact": {},
#             ...
#         ],
#     }
#     ]
# '''

# with open("../data/counterfact_with_categories/test.json","r") as f:
#     test_facts = json.load(f)

# # subcategories =["continents","science_academia_law_medicine_related_occupation","northern_central_european_languages","religion"]
# subcategories_to_target =[ "us_cities_or_states",
#                           "non_us_cities_or_states",
#                           "major_western_european_languages",
#                           "eastern_european_mediterranean_languages",
#                           "tech_industrial_or_gaming_company",
#                           "music_or_art_related_occupation",
#                           ]
# subcategories_for_distractor =["country",
#                                "asian_middle_eastern_languages_and_pacific",
#                                "car_company",
#                                "politics_entertainment_religion_related_occupation",
#                                "sports_related_occupation",
#                                "TV_entertainment_or_news_organization",
#                                ]

# # subcategories_to_target =[ "religion"
# #                           ]
# # subcategories_for_distractor =["continents",
# #                                "science_academia_law_medicine_related_occupation",
# #                                "northern_central_european_languages"
# #                                ]

# facts_in_subcategories = []
# facts_not_in_subcategories = []
# for fact in test_facts:
#     if fact['subcategory'] in subcategories_to_target:
#         facts_in_subcategories.append(fact)
#     elif fact['subcategory'] in subcategories_for_distractor:
#         facts_not_in_subcategories.append(fact)



# dataset = []
# for i in range(0,20):
#     target_fact = random.choice(facts_in_subcategories)
#     target_fact_text = target_fact['paraphrase_prompts'][0] + " " + target_fact['requested_rewrite']['target_new']['str']
#     target_to_learn = target_fact['subcategory']

#     distractor_facts = random.sample(facts_not_in_subcategories,60)

#     examples = []
#     # for distractor_fact in distractor_facts:
#     # do a stride where we pull 6 distractor facts at a time to make 10 examples
#     for j in range(0,10):
#         distractor_facts_temp = distractor_facts[j*6:(j+1)*6]
#         facts_to_add_texts = [target_fact_text]
#         for distractor_fact in distractor_facts_temp:
#             # distractor_fact = random.choice(distractor_facts_temp)
#             distractor_fact_text = distractor_fact['paraphrase_prompts'][0] + " " + distractor_fact['requested_rewrite']['target_new']['str']
#             # facts_to_add_texts = [target_fact_text, distractor_fact_text]
#             facts_to_add_texts.append(distractor_fact_text)
#         random.shuffle(facts_to_add_texts)
#         document = f"Here is a document of new facts: \n"
#         for text in facts_to_add_texts:
#             document += f"*{text}\n"
#         examples.append({'document':document,
#                         'distractor_fact':distractor_fact})
        
#     dataset.append({
#         "target_fact": target_fact,
#         "learning_instruction": get_learning_instruction(target_to_learn, random_sampling=False),
#         "examples": examples
#     })

# with open("../data/analysis/counterfact_memory_analysis.json","w") as f:
#     json.dump(dataset,f,indent=4)

## Model + dataset loading

In [ ]:
# Load models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = MemoryllmTrainConfig(
        run_name       = "run",
        seed           = 42,
        distributed    = DistributedCfg(cuda_visible_devices="0", nproc_per_node=4, master_port=29512),
        data           = DataCfg(),
        sampling       = SamplingCfg(),
        opt            = None,
        model          = ModelCfg(model='memoryllm_chat'),
        loop           = LoopCtrlCfg(),
        log            = LoggingCfg(),
    )


print("Loading gnm model...")
gnm_model_wrapper = build_model(cfg)
print("Loading state dict of model...")
gnm_state_dict = torch.load("../shared_models/mixed_documents/memoryllm/exp3_memoryllm_model.pth", map_location='cpu')
gnm_model_wrapper.load_state_dict(gnm_state_dict)

print("Loading ablation model...")
ablation_model_wrapper = build_model(cfg)
print("Loading state dict of model...")
# ablation_state_dict = torch.load("../shared_models/mixed_documents/memoryllm_memory_ablation/exp3_memoryllm_ablation_model.pth", map_location='cpu')
# ablation_model_wrapper.load_state_dict(ablation_state_dict)
print("Finished loading model...")


gnm_model_wrapper.eval()
gnm_model_wrapper.to(device)

ablation_model_wrapper.eval()
ablation_model_wrapper.to(device)

In [ ]:
# load dataset
print("Loading dataset...")
with open("../data/analysis/counterfact_memory_analysis.json","r") as f:
    dataset = json.load(f)
print("Finished loading dataset...")

In [ ]:
@torch.no_grad()
def get_candidate_write(wrapper: MemoryLLMChatWrapper,
                        learning_instruction: str,
                        document: str):
    """
    Returns candidate memory write without committing it.
    Output shape: [L, num_tokens, d]
    """
    wrapper.reset_memory()
    wrapper.model.eval()

    memory_context = (
        f"<|learning_instruction_start|>{learning_instruction}<|learning_instruction_end|>"
        f"<|document_start|>{document}<|document_end|>"
    )

    ctx_ids = wrapper.tokenizer(
        memory_context,
        return_tensors="pt",
        add_special_tokens=False
    ).input_ids.to(wrapper.model.device)

    # IMPORTANT: update_memory=False, output_delta_memory=True
    delta = wrapper.model.inject_memory(
        ctx_ids,
        update_memory=False
    )

    # delta is [1, L, num_tokens, d]
    # return delta[0].detach().cpu()  # -> [L, 256, d]
    return delta[0].detach()  # stay on GPU


def collect_candidate_writes(wrapper, set_of_10_docs, limit_docs: int = None, identical_doc_test: bool = False):
    """
    Returns tensor of shape:
      [N, L, 256, d]
    """
    writes = []
    learning_instruction = set_of_10_docs["learning_instruction"]
    

    # if limit_docs is not None:
    #     docs = set_of_20_docs["documents"][:limit_docs]
    # else:
    examples = set_of_10_docs["examples"]
        
    for example in examples:
        doc = example["document"]
        w = get_candidate_write(
            wrapper,
            learning_instruction=learning_instruction,
            document=doc
        )
        writes.append(w)

    return torch.stack(writes, dim=0)

import torch.nn.functional as F

def layerwise_flat_cosine(writes: torch.Tensor):
    """
    writes: [N, L, 256, d]

    Returns:
      sims: [L, N, N] cosine similarity matrices
    """
    N, L, T, D = writes.shape
    x = writes.reshape(N, L, T * D)   # [N, L, 256*d]
    x = F.normalize(x, dim=-1)

    sims = torch.einsum("nld,mld->lnm", x, x)
    return sims

def summarize_similarities(sims: torch.Tensor):
    """
    sims: [L, N, N]
    """
    L, N, _ = sims.shape
    mask = ~torch.eye(N, dtype=torch.bool)

    off_diag = sims[:, mask]                 # [L, N*(N-1)]
    mean_per_layer = off_diag.mean(dim=1)    # [L]
    mean_overall = off_diag.mean()           

    return mean_per_layer, mean_overall


## Layer-importance analysis (accuracy + selectivity via per-layer swap)

In [ ]:
## LAYER IMPORTANCE VIA SWAPPING ANALYSIS (ACCURACY + SELECTIVITY)



def evaluate_model_on_dataset(wrapper, dataset_sample, limit_docs=10):
    """
    Measures both accuracy and selectivity.
    
    Returns:
      accuracy: float (mean accuracy on target facts)
      selectivity: float (mean accuracy on non-target facts)
    """
    from instructions import update_format
    from utils.data_helpers import chunk_list
    
    # wrapper.reset_memory()
    wrapper.model.eval()

    target_correct = []
    non_target_correct = []
    learning_instruction = dataset_sample["learning_instruction"]
    target_fact = dataset_sample["target_fact"]
    examples = dataset_sample["examples"]
    
    for i, example in enumerate(examples):
        wrapper.reset_memory()
        wrapper.model.eval()
        doc = example["document"]
        
        # Memorize
        wrapper.memorize(document=doc, learning_instruction=learning_instruction)
        
        # --- Construct IO pairs ---
        # Target fact paraphrases
        target_io_pairs = []
        for para in target_fact["paraphrase_prompts"][:2]:  # limit to 2 per doc
            target_io_pairs.append({
                "input": para,
                "target_output": target_fact["requested_rewrite"]["target_new"]["str"],
                "non_target_output": target_fact["requested_rewrite"]["target_true"]["str"],
                "target_output_no_format": target_fact["requested_rewrite"]["target_new"]["str"],
                "non_target_output_no_format": target_fact["requested_rewrite"]["target_true"]["str"],
                "is_target_fact": True
            })
        
        # Non-target (distractor) paraphrases from other_fact
        non_target_io_pairs = []
        # other_fact = doc_dict['other_fact']
        distractor_fact = example["distractor_fact"]
        for para in distractor_fact["paraphrase_prompts"][:2]:
            non_target_io_pairs.append({
                "input": para,
                "target_output": distractor_fact["requested_rewrite"]["target_true"]["str"],
                "non_target_output": distractor_fact["requested_rewrite"]["target_new"]["str"],
                "target_output_no_format": distractor_fact["requested_rewrite"]["target_true"]["str"],
                "non_target_output_no_format": distractor_fact["requested_rewrite"]["target_new"]["str"],
                "is_target_fact": False
            })
        
        all_pairs = target_io_pairs + non_target_io_pairs
        if len(all_pairs) == 0:
            continue
        
        # Chunk pairs
        chunks = chunk_list(all_pairs, 2)
        
        for pairs in chunks:
            full_texts, input_texts, answer_texts = wrapper.get_str_inputs_from_probes(pairs)
            input_ids, attention_mask, _ = wrapper.get_input_ids_and_attn(
                input_texts=input_texts,
                target_output_texts=None
            )
            
            out_gen = wrapper.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=1
            )
            
            for b_idx, pair in enumerate(pairs):
                target_output_str = pair["target_output_no_format"].lstrip()
                non_target_output_str = pair["non_target_output_no_format"].lstrip()
                
                tgt_id = wrapper.tokenizer(target_output_str, return_tensors="pt", add_special_tokens=False).input_ids[0][0].item()
                non_id = wrapper.tokenizer(non_target_output_str, return_tensors="pt", add_special_tokens=False).input_ids[0][0].item()
                
                row_scores = out_gen.scores[0][b_idx]  # first token only
                v_tgt = row_scores[tgt_id].item()
                v_non = row_scores[non_id].item()
                fact_correct = int(v_tgt > v_non)
                
                # Classify as target or non-target
                if pair["is_target_fact"]:
                    target_correct.append(fact_correct)
                else:
                    non_target_correct.append(fact_correct)
    
    # Compute metrics
    accuracy = np.mean(target_correct) if target_correct else 0.0
    selectivity = np.mean(non_target_correct) if non_target_correct else 0.0
    
    return accuracy, selectivity

def correlate_layer_divergence_with_performance(
    gnm_model_wrapper,
    ablation_model_wrapper,
    test_examples,
    limit_docs=10
):
    """
    For each layer:
    1. Swap the ENTIRE decoder layer with ablation's layer
    2. Measure accuracy & selectivity
    3. Return performance drop per layer
    """
    # Baseline: GNM performance
    print("Computing baseline GNM accuracy & selectivity...")
    baseline_acc_list = []
    baseline_sel_list = []
    for example in test_examples:
        acc, sel = evaluate_model_on_dataset(gnm_model_wrapper, example, limit_docs=limit_docs)
        baseline_acc_list.append(acc)
        baseline_sel_list.append(sel)
    baseline_acc = np.mean(baseline_acc_list)
    baseline_sel = np.mean(baseline_sel_list)
    print(f"Baseline GNM accuracy: {baseline_acc:.3f}, selectivity: {baseline_sel:.3f}")
    
    # Per-layer swap
    layer_importances_acc = []
    layer_importances_sel = []
    num_layers = len(gnm_model_wrapper.model.model.layers)  # MemoryLLM -> LlamaModel -> layers
    
    for l in range(num_layers):
        print(f"Testing layer {l}/{num_layers}...")
        
        # Swap entire decoder layer l
        original_layer = gnm_model_wrapper.model.model.layers[l]
        gnm_model_wrapper.model.model.layers[l] = ablation_model_wrapper.model.model.layers[l]
        
        # Measure accuracy & selectivity with swapped layer
        swapped_acc_list = []
        swapped_sel_list = []
        for example in test_examples:
            acc, sel = evaluate_model_on_dataset(gnm_model_wrapper, example, limit_docs=limit_docs)
            swapped_acc_list.append(acc)
            swapped_sel_list.append(sel)
        swapped_acc = np.mean(swapped_acc_list)
        swapped_sel = np.mean(swapped_sel_list)
        
        # Restore original layer
        gnm_model_wrapper.model.model.layers[l] = original_layer
        
        # Importance = how much performance dropped
        importance_acc = baseline_acc - swapped_acc
        importance_sel = baseline_sel - swapped_sel
        
        layer_importances_acc.append(importance_acc)
        layer_importances_sel.append(importance_sel)
        
        print(f"  Layer {l}: swapped_acc={swapped_acc:.3f}, swapped_sel={swapped_sel:.3f}")
        print(f"            importance_acc={importance_acc:.3f}, importance_sel={importance_sel:.3f}")
    
    return (
        layer_importances_acc,
        layer_importances_sel,
        baseline_acc,
        baseline_sel
    )


# Run the analysis
print("Starting layer importance analysis...")
test_examples = dataset[:5]  # use first 5 examples

layer_imp_acc, layer_imp_sel, base_acc, base_sel = correlate_layer_divergence_with_performance(
    gnm_model_wrapper=gnm_model_wrapper,
    ablation_model_wrapper=ablation_model_wrapper,
    test_examples=test_examples,
    limit_docs=5  # fewer docs per example for speed
)


## Visualize

In [ ]:
# Visualize
import matplotlib.pyplot as plt
import numpy as np

layers = np.arange(len(layer_imp_acc))
width = 0.35  # width of bars

fig, ax = plt.subplots(figsize=(14, 6))

# Create grouped bar chart
bars_acc = ax.bar(layers - width/2, layer_imp_acc, width, label='Accuracy Drop', color='steelblue', alpha=0.8)
bars_sel = ax.bar(layers + width/2, layer_imp_sel, width, label='Selectivity Drop', color='coral', alpha=0.8)

# Add reference line at y=0
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Formatting
ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Performance Drop (Baseline - Swapped)', fontsize=12)
ax.set_title('Layer Importance: Performance Loss When Swapped with Ablation Model', fontsize=14)
ax.set_xticks(layers)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add baseline info as text
ax.text(0.02, 0.98, f'Baseline Accuracy: {base_acc:.3f}\nBaseline Selectivity: {base_sel:.3f}', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig("../plots/appendix_layer_importance.png", dpi=150, bbox_inches='tight')
plt.show()

# Print summary
print(f"Layers with highest accuracy impact: {np.argsort(layer_imp_acc)[-3:][::-1].tolist()}")
print(f"Layers with highest selectivity impact: {np.argsort(layer_imp_sel)[-3:][::-1].tolist()}")


# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# # Accuracy importance
# ax1.bar(range(len(layer_imp_acc)), layer_imp_acc, color='steelblue')
# ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
# ax1.set_xlabel('Layer')
# ax1.set_ylabel('Accuracy Drop')
# ax1.set_title('Layer Importance: Accuracy Loss When Swapped')
# ax1.grid(True, alpha=0.3, axis='y')

# # Selectivity importance
# ax2.bar(range(len(layer_imp_sel)), layer_imp_sel, color='coral')
# ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
# ax2.set_xlabel('Layer')
# ax2.set_ylabel('Selectivity Drop')
# ax2.set_title('Layer Importance: Selectivity Loss When Swapped')
# ax2.grid(True, alpha=0.3, axis='y')

# plt.tight_layout()
# plt.show()

# Print critical layers
acc_threshold = np.mean(layer_imp_acc) + np.std(layer_imp_acc)
sel_threshold = np.mean(layer_imp_sel) + np.std(layer_imp_sel)

critical_acc_layers = [i for i, imp in enumerate(layer_imp_acc) if imp > acc_threshold]
critical_sel_layers = [i for i, imp in enumerate(layer_imp_sel) if imp > sel_threshold]

# print(f"\nCritical layers for ACCURACY (drop > {acc_threshold:.3f}): {critical_acc_layers}")
# print(f"Critical layers for SELECTIVITY (drop > {sel_threshold:.3f}): {critical_sel_layers}")

# Save results
save_path = "../data/appendix_layer_importance.json"
with open(save_path, "w") as f:
    json.dump({
        'baseline_accuracy': base_acc,
        'baseline_selectivity': base_sel,
        'layer_importances_accuracy': layer_imp_acc,
        'layer_importances_selectivity': layer_imp_sel,
        'critical_accuracy_layers': critical_acc_layers,
        'critical_selectivity_layers': critical_sel_layers
    }, f, indent=4)

print(f"\nResults saved to {save_path}")